# Municipal 311 Assistant — Adapter Sprint (QLoRA + DPO under a 15GB ceiling)

Rewrites messy resident 311 complaints into structured dispatch tickets.

This notebook is the **Colab walkthrough and orchestrator**. The reusable logic (data
generation, the token-length audit, QLoRA training + the deliberate OOM/fix, the
dual-adapter DPO reference design, and evaluation) lives in `src/`, so it can be run,
tested, and explained independently of this notebook. Run cells top to bottom on a T4
runtime; every number in the results section is produced by the code that runs it, in
`logs/experiment_log.json` and `outputs/eval/eval_*.json` -- nothing here is pre-filled.

**Status of this version:** data generation and the preference dataset have been run
already (see `data/`). QLoRA training, the deliberate OOM, and DPO have **not** been run
yet -- `configs/qlora_config.json` intentionally has `max_seq_len: null` pending the
token-length audit in Section 2, which requires Hub access this notebook has in Colab.

## 0. Setup

In [ ]:
!pip install -q -U -r requirements.txt


In [ ]:
import sys, os, json
sys.path.insert(0, "src")

import pandas as pd
import torch

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
QLORA_CONFIG_PATH = "configs/qlora_config.json"
DPO_CONFIG_PATH = "configs/dpo_config.json"
LOG_PATH = "logs/experiment_log.json"

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE -- select a T4 runtime")


## 1. Synthetic 311 complaint + ticket data

Already generated offline by `src/generate_instruction_data.py` -- 360 examples, ~15% with
no street address (`address_or_null: null` in the ground truth), split into train/val/held-out
test. Rerun the cell below only if you want a fresh sample.

In [ ]:
!python src/generate_instruction_data.py --n 360 --seed 42 --out-dir data


In [ ]:
with open("data/generation_summary.json") as f:
    print(json.dumps(json.load(f), indent=2))


## 2. Token-length audit -- run BEFORE setting max_seq_len

`configs/qlora_config.json` has `max_seq_len: null` on purpose. Do not hardcode a
sequence length assumption; measure the real tokenized lengths of the generated data
first, review the recommendation, and only then approve a value.

In [ ]:
!python src/audit_instruction_data.py --data data/sft_train.jsonl --model {MODEL_NAME} --out data/token_length_audit.json


In [ ]:
with open("data/token_length_audit.json") as f:
    audit_report = json.load(f)
print(json.dumps(audit_report, indent=2))
print()
print("Recommended max_seq_len:", audit_report["recommended_max_seq_len"])
print("Review this against the printed distribution, then run the next cell to write your")
print("APPROVED value into configs/qlora_config.json. Do not skip the review step.")


In [ ]:
# Human-in-the-loop approval step: set this to the value you approve (usually, but not
# necessarily, audit_report["recommended_max_seq_len"]) and it gets written into the config
# that train_qlora.py reads. train_qlora.py refuses to run while this is still null.
APPROVED_MAX_SEQ_LEN = None  # <-- set this after reviewing the audit above, then run this cell

if APPROVED_MAX_SEQ_LEN is None:
    print("APPROVED_MAX_SEQ_LEN is still None -- set it above before continuing.")
else:
    with open(QLORA_CONFIG_PATH) as f:
        qlora_cfg = json.load(f)
    qlora_cfg["max_seq_len"] = APPROVED_MAX_SEQ_LEN
    qlora_cfg["max_seq_len_status"] = f"APPROVED by human review of data/token_length_audit.json (recommended {audit_report['recommended_max_seq_len']})"
    with open(QLORA_CONFIG_PATH, "w") as f:
        json.dump(qlora_cfg, f, indent=2)

    with open(DPO_CONFIG_PATH) as f:
        dpo_cfg = json.load(f)
    dpo_cfg["max_prompt_length"] = APPROVED_MAX_SEQ_LEN
    dpo_cfg["max_length"] = APPROVED_MAX_SEQ_LEN + 64
    dpo_cfg["max_length_status"] = f"Derived from approved qlora max_seq_len={APPROVED_MAX_SEQ_LEN}"
    with open(DPO_CONFIG_PATH, "w") as f:
        json.dump(dpo_cfg, f, indent=2)
    print(f"Wrote max_seq_len={APPROVED_MAX_SEQ_LEN} into {QLORA_CONFIG_PATH} and {DPO_CONFIG_PATH}")


## 3. Preference dataset for DPO

Already generated by `src/generate_preference_data.py`. Every complaint with no address in
the ground truth gets a `hallucinate_address` rejection (chosen preserves `null`, rejected
invents a street); complaints with a real address get one of the other corruption types
(dropped field, wrong category, malformed JSON, or a wrong/hallucinated address).

In [ ]:
!python src/generate_preference_data.py --data data/sft_train.jsonl --out data/dpo_pairs.jsonl --max-pairs 180


In [ ]:
with open("data/dpo_pairs_summary.json") as f:
    print(json.dumps(json.load(f), indent=2))


## 4. Baseline: evaluate the raw base model

Uses `src/evaluate.py` directly (no adapter) on the held-out test split. This includes the
`address_null_recall` / `address_null_precision` metrics that specifically track the
missing-address hallucination behavior DPO is meant to fix.

In [ ]:
from resource_monitor import profile_run
from evaluate import load_model_for_eval, evaluate_model, load_jsonl

test_examples = load_jsonl("data/sft_test.jsonl")

def run_base_eval():
    model, tokenizer = load_model_for_eval(MODEL_NAME, adapter_path=None)
    return evaluate_model(model, tokenizer, test_examples, tag="base", out_dir="outputs/eval")

base_metrics, _ = profile_run("baseline_eval", run_base_eval, config={"model": MODEL_NAME}, log_path=LOG_PATH)


## 5. QLoRA supervised fine-tuning

Requires `configs/qlora_config.json` to have an approved `max_seq_len` (Section 2). Uses
`src/train_qlora.py`, which verifies the proposed LoRA target modules against this model's
actual linear layer names before attaching adapters, and prints the exact trainable
parameter count and percentage so the capacity choice can be defended with real numbers.

In [ ]:
!python src/train_qlora.py --config {QLORA_CONFIG_PATH} --data-dir data --out-dir outputs --log-path {LOG_PATH}


## 6. Deliberate OOM, diagnosis, and fix

Pushes batch size and sequence length up with gradient checkpointing off (same 0.5B model)
to genuinely exceed the T4's memory, captures the real `CUDA OutOfMemoryError` traceback to
`logs/oom_traceback.txt`, then reruns a single step at the safe config to show the fix
works. If this exact config does not OOM on your GPU allocation, raise
`deliberate_oom.unsafe_batch_size` / `unsafe_seq_len` in `configs/qlora_config.json` and
rerun -- the script will say so rather than reporting a false success.

In [ ]:
!python src/train_qlora.py --config {QLORA_CONFIG_PATH} --data-dir data --out-dir outputs --log-path {LOG_PATH} --run-oom-demo


### Diagnosis (fill in from the actual `logs/oom_traceback.txt` produced above)

- **What failed:** `torch.cuda.OutOfMemoryError` raised during `<paste the exact call from your traceback>`.
- **Root cause:** with gradient checkpointing off, activation memory scales with
  `batch_size x seq_len^2 x num_layers`; at the unsafe config this exceeds the ~15GB T4
  budget even though the 4-bit base model itself only needs a few hundred MB.
- **Why the fix works:** re-enabling gradient checkpointing trades recomputation for not
  storing per-layer activations; the safe config's small batch + gradient accumulation
  reaches the same effective batch size at a fraction of the peak activation memory.
- **Evidence:** compare the `peak_allocated_gb` / `peak_reserved_gb` fields for the
  `deliberate_oom_attempt` (if captured) and `post_fix_step` records in
  `logs/experiment_log.json`.

## 7. Evaluate the QLoRA model

In [ ]:
def run_qlora_eval():
    model, tokenizer = load_model_for_eval(MODEL_NAME, adapter_path="outputs/qlora_adapter")
    return evaluate_model(model, tokenizer, test_examples, tag="qlora", out_dir="outputs/eval")

qlora_metrics, _ = profile_run("qlora_eval", run_qlora_eval, config={"adapter": "outputs/qlora_adapter"}, log_path=LOG_PATH)


## 8. DPO on top of the QLoRA model

Uses `src/train_dpo.py`, which loads the SFT adapter twice onto one shared 4-bit base
(`policy`, trainable; `reference`, frozen immediately after loading) so the reference
represents the instruction-tuned state before DPO without paying for a second full model
copy. The script verifies this against the installed TRL/PEFT version and prints exactly
which parameters are trainable vs frozen before training starts.

In [ ]:
!python src/train_dpo.py --dpo-config {DPO_CONFIG_PATH} --qlora-config {QLORA_CONFIG_PATH} --data data/dpo_pairs.jsonl --out-dir outputs --log-path {LOG_PATH}


## 9. Evaluate the DPO model

In [ ]:
def run_dpo_eval():
    model, tokenizer = load_model_for_eval(MODEL_NAME, adapter_path="outputs/dpo_adapter")
    return evaluate_model(model, tokenizer, test_examples, tag="dpo", out_dir="outputs/eval")

dpo_metrics, _ = profile_run("dpo_eval", run_dpo_eval, config={"adapter": "outputs/dpo_adapter"}, log_path=LOG_PATH)


## 10. Results

Both tables are built from `logs/experiment_log.json` and `outputs/eval/eval_*.json` --
run the notebook top to bottom first; an empty table here means a stage hasn't run yet, not
zero.

In [ ]:
with open(LOG_PATH) as f:
    perf_records = json.load(f)
pd.DataFrame(perf_records)


In [ ]:
compare_rows = []
for tag in ["base", "qlora", "dpo"]:
    path = f"outputs/eval/eval_{tag}.json"
    if os.path.exists(path):
        with open(path) as f:
            compare_rows.append(json.load(f)["metrics"])
pd.DataFrame(compare_rows)


## 11. Limitations

- All complaint/ticket data is synthetically generated by templates, not real 311 records.
- The preference dataset's `rejected` examples are rule-corrupted rather than mined from
  the SFT model's own failures.
- `logs/experiment_log.json` and `outputs/eval/eval_*.json` are the source of truth for
  every number this notebook reports. If a section wasn't (re)run, don't quote its numbers
  as current.